# Stage 4 — CNN Patch-Level Source Classification

This notebook implements the final thesis stage: **patch-level source classification from monthly Sentinel-5P-derived methane raster products**.

The task is **binary patch classification**, not plume segmentation, not source localisation, and not emission-rate estimation. A patch is labelled as source-associated when it overlaps a monthly Carbon Mapper source target constructed from a named source with at least one attributed methane plume-event record in the same month. Strict no-source patches are selected using known-source exclusion, bivariate class screening, and enhancement-area screening.

## Predictor channels

Each sample is a non-overlapping `64 × 64` raster patch with five channels:

1. monthly mean Sentinel-5P XCH4;
2. 3° local-background anomaly;
3. 3° continuous standardised enhancement `Z`;
4. 5° local-background anomaly;
5. 5° continuous standardised enhancement `Z`.

The bivariate raster is **not** a predictor. It is used only to construct conservative no-source labels.

## Main workflow

1. Locate monthly raster inputs and Carbon Mapper source/plume tables.
2. Build monthly source labels from `point_source_name` and `scene_timestamp`.
3. Rasterise monthly source targets as Gaussian label masks.
4. Build a natural-distribution binary patch catalogue.
5. Estimate channel normalisation statistics from training patches only.
6. Train a five-branch CNN with temporally separated train/validation/test periods.
7. Select checkpoint and threshold using validation data only.
8. Evaluate on held-out test months.
9. Export metrics, predictions, checkpoint files, and monthly confusion-map figures.

## Thesis interpretation

The output probability is a **source-associated patch probability under the constructed labels**. It must not be interpreted as a facility-level detection probability, plume detection, source localisation, or emission-rate estimate.


## 1. Install And Mount Drive


In [ ]:
!pip -q install rasterio tqdm scikit-learn matplotlib


In [ ]:
# Configure the project folder.
#
# Expected project folder structure:
# CNN_point_source/
# ├── sources_plumes/
# │   └── plumes_with_point_source_name.csv
# ├── mean_xch4/
# ├── anomaly_3deg/
# ├── enhancement_3deg/
# ├── anomaly_5deg/
# ├── enhancement_5deg/
# ├── bivariate/
# ├── prepared_data/
# └── models/
#
# In Google Colab, the default location is:
# /content/drive/MyDrive/CNN_point_source
#
# Outside Colab, set the environment variable CNN_PROJECT_DIR, or edit
# DRIVE_PROJECT_DIR below.

import os
import subprocess
from pathlib import Path

try:
    from google.colab import drive
    IN_COLAB = True
except ModuleNotFoundError:
    IN_COLAB = False

DRIVE_PROJECT_DIR = os.environ.get(
    "CNN_PROJECT_DIR",
    "/content/drive/MyDrive/CNN_point_source",
)
LOCAL_PROJECT_DIR = os.environ.get(
    "CNN_LOCAL_PROJECT_DIR",
    "/content/CNN_point_source",
)

if IN_COLAB:
    drive.mount("/content/drive")
    os.makedirs(LOCAL_PROJECT_DIR, exist_ok=True)

    # Copy Drive data locally for faster raster reads. If rsync fails, the
    # notebook falls back to using Drive directly.
    rsync_result = subprocess.run(
        [
            "rsync",
            "-ah",
            "--info=progress2",
            "--partial",
            f"{DRIVE_PROJECT_DIR}/",
            f"{LOCAL_PROJECT_DIR}/",
        ],
        check=False,
    )

    if rsync_result.returncode == 0:
        PROJECT_DIR = LOCAL_PROJECT_DIR
        print("Using local project copy:", PROJECT_DIR)
    else:
        PROJECT_DIR = DRIVE_PROJECT_DIR
        print("rsync failed; using Drive folder directly:", PROJECT_DIR)
else:
    PROJECT_DIR = DRIVE_PROJECT_DIR
    print("Running outside Colab.")
    print("Using project folder:", PROJECT_DIR)

PROJECT_DIR = str(Path(PROJECT_DIR))


## Data prerequisites

This notebook assumes that Stages 2 and 3 have already produced the monthly raster products required for CNN training.

Required inputs:

- Carbon Mapper plume/source table with `point_source_name`, coordinates, gas, and timestamps;
- monthly mean XCH4 rasters;
- 3° anomaly rasters;
- 3° continuous standardised-enhancement rasters;
- 5° anomaly rasters;
- 5° continuous standardised-enhancement rasters;
- monthly bivariate class rasters for conservative negative screening.

The notebook checks raster availability before building labels and patch catalogues. Large GeoTIFFs and model checkpoints are stored in the external thesis dataset described in `docs/data-availability.md`.


## 2. Imports And Configuration


In [ ]:
import math
import random
import re
from dataclasses import dataclass
from datetime import datetime, timezone
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import rasterio
from rasterio.enums import Resampling
from rasterio.windows import Window
from sklearn.metrics import (
    accuracy_score,
    average_precision_score,
    confusion_matrix,
    ConfusionMatrixDisplay,
    f1_score,
    precision_score,
    recall_score,
    roc_auc_score,
)
from tqdm.auto import tqdm

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, Dataset


@dataclass
class CFG:
    project_dir: str = PROJECT_DIR
    years: tuple = (2023, 2024, 2025)
    patch_size: int = 64
    batch_size: int = 32
    num_workers: int = 0
    epochs: int = 100
    early_stopping_patience: int = 8
    early_stopping_min_delta: float = 1e-4
    lr: float = 1e-4
    weight_decay: float = 2e-2
    label_smoothing: float = 0.0
    grad_clip_norm: float = 1.0
    base_channels: int = 16
    dropout: float = 0.4

    # Non-overlapping catalog logic.
    # Keep all positives and all strict negatives. Imbalance is handled
    # only by the training loss, not by dropping samples.
    use_all_strict_negatives: bool = True
    negative_target_max: float = 1e-6
    gaussian_sigma_pixels: float = 3.0

    # Strict negative logic.
    # A negative patch must have bivariate class 1 at the patch center.
    bivariate_center_class: int = 1
    bivariate_nodata_values: tuple = (0,)
    enhancement_area_threshold: float = 2.0
    max_enhancement_area_fraction: float = 0.20

    # Manual positive class weight.
    pos_weight: float = 2.5

    # Online flip/rotation augmentation is intentionally off. The safe
    # augmentation below shifts the patch center and re-checks the label rules.
    augment_training: bool = False
    augment_flips: bool = False
    augment_rot90: bool = False
    shift_augment_training: bool = False
    train_shift_augmentations_per_patch: int = 1
    train_shift_max_pixels: int = 16
    train_shift_max_tries_per_patch: int = 30

    # Use every natural training sample exactly once per epoch.
    # Mild positive weighting is applied only in the training loss.
    train_balanced_sampler: bool = False
    run_tag: str = "natural_all_samples_posw2p5_drop04"
    # Leave blank for a new timestamped run. Set an existing folder
    # name here only when intentionally resuming/evaluating that run.
    run_id: str = "natural_all_samples_posw2p5_drop04_20260606T164750_944615Z"

    stats_windows: int = 512
    rebuild_labels: bool = False
    rebuild_targets: bool = False
    rebuild_patch_catalog: bool = False
    seed: int = 42
    device: str = "cuda" if torch.cuda.is_available() else "cpu"


cfg = CFG()
RUN_ID = cfg.run_id.strip() or (
    f"{cfg.run_tag}_"
    f"{datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%S_%fZ')}"
)
random.seed(cfg.seed)
np.random.seed(cfg.seed)
torch.manual_seed(cfg.seed)
print("CUDA available:", torch.cuda.is_available())
print("Run ID:", RUN_ID)
print(cfg)


## Check 3° and 5° raster availability

This preflight check verifies the required 5-degree anomaly and enhancement rasters for every month in 2023-2025 before manifest and catalogue construction. The enhancement role accepts either raw `enhancement_..._5deg...` rasters or `enhancementGE2_..._5deg...` mask rasters.


In [ ]:
# Check that all monthly 3-degree and 5-degree anomaly/enhancement rasters are present and readable.
# This cell only checks files; it does not resample or modify rasters.

from pathlib import Path
import re
import pandas as pd
import rasterio

CHECK_ANOM3_RE = re.compile(r"^anomaly_(?P<year>20\d{2})_(?P<month>\d{2})_3deg_CONUSbuf2p5deg\.tif$")
CHECK_ENH3_RE = re.compile(r"^enhancement_(?P<year>20\d{2})_(?P<month>\d{2})_3deg_CONUSbuf2p5deg\.tif$")
CHECK_ANOM5_RE = re.compile(r"^anomaly_(?P<year>20\d{2})_(?P<month>\d{2})_5deg_CONUSbuf2p5deg\.tif$")
CHECK_ENH5_RE = re.compile(
    r"^enhancement(?:GE2)?_(?P<year>20\d{2})_(?P<month>\d{2})_5deg(?:_continuous)?_CONUSbuf2p5deg\.tif$"
)


def check_raster_search_roots(project_dir):
    roots = []

    def add_root(path):
        path = Path(path)
        try:
            exists = path.exists()
        except OSError as exc:
            print(f"Skipping unavailable search root: {path} ({exc})")
            return
        if exists and path not in roots:
            roots.append(path)

    add_root(project_dir)
    for root in globals().get("EXTRA_RASTER_ROOTS", []):
        add_root(root)
    add_root("/content/CNN_point_source")
    add_root("/content/drive/MyDrive/CNN_point_source")
    add_root("/content/drive/MyDrive/CNN_pixel_level")
    return roots


def inspect_raster(path):
    try:
        with rasterio.open(path) as src:
            return {
                "readable": True,
                "width": src.width,
                "height": src.height,
                "crs": str(src.crs),
                "nodata": src.nodata,
                "error": "",
            }
    except Exception as exc:
        return {
            "readable": False,
            "width": None,
            "height": None,
            "crs": None,
            "nodata": None,
            "error": repr(exc),
        }


project_for_check = Path(cfg.project_dir)
roots = check_raster_search_roots(project_for_check)
print("Searching for 3-degree and 5-degree rasters in:")
for root in roots:
    print(" -", root)

found = {}
for root in roots:
    try:
        tif_paths = list(root.rglob("*.tif"))
    except OSError as exc:
        print(f"Skipping unavailable search root during scan: {root} ({exc})")
        continue

    for path in tif_paths:
        for role, regex in {
            "anomaly_3deg": CHECK_ANOM3_RE,
            "enhancement_3deg": CHECK_ENH3_RE,
            "anomaly_5deg": CHECK_ANOM5_RE,
            "enhancement_5deg": CHECK_ENH5_RE,
        }.items():
            match = regex.match(path.name)
            if not match:
                continue
            year = int(match.group("year"))
            month = int(match.group("month"))
            if year in cfg.years:
                found.setdefault((year, month, role), []).append(path)

rows = []
required_roles = ["anomaly_3deg", "enhancement_3deg", "anomaly_5deg", "enhancement_5deg"]
for year in cfg.years:
    for month in range(1, 13):
        for role in required_roles:
            paths = found.get((year, month, role), [])
            paths = sorted(paths, key=lambda p: (0 if "_continuous_" in p.name else 1, str(p)))
            chosen = paths[0] if paths else None
            meta = inspect_raster(chosen) if chosen else {
                "readable": False,
                "width": None,
                "height": None,
                "crs": None,
                "nodata": None,
                "error": "missing",
            }
            rows.append({
                "year": year,
                "month": month,
                "ym": f"{year}-{month:02d}",
                "role": role,
                "path_count": len(paths),
                "path": str(chosen) if chosen else "",
                **meta,
            })

check_multiscale_df = pd.DataFrame(rows)
display(check_multiscale_df)

missing = check_multiscale_df[~check_multiscale_df["readable"]]
duplicates = check_multiscale_df[check_multiscale_df["path_count"] > 1]

readable_months = (
    check_multiscale_df[check_multiscale_df["readable"]]
    .groupby(["year", "month"])["role"]
    .nunique()
    .reset_index(name="readable_required_roles")
)
complete_months = int((readable_months["readable_required_roles"] == len(required_roles)).sum())

print(f"\nComplete readable multi-scale months found: {complete_months} / 36")

print("\nMissing or unreadable multi-scale rasters:")
if missing.empty:
    print([])
else:
    display(missing[["ym", "role", "error"]])

print("\nDuplicate readable multi-scale rasters:")
if duplicates.empty:
    print([])
else:
    display(duplicates[["ym", "role", "path_count", "path"]])


## 3. Find XCH4, anomaly, enhancement, and bivariate screening rasters


In [ ]:
project = Path(cfg.project_dir)
source_dir = project / "sources_plumes"
plumes_csv = source_dir / "plumes_with_point_source_name.csv"
sources_csv = source_dir / "matched_point_sources_with_matched_plumes.csv"

experiment = f"multiscale_methane_strict_negative_2023_2025_ps{cfg.patch_size}"
prepared_dir = project / "prepared_data"
target_dir = prepared_dir / f"monthly_targets_{experiment}"
labels_path = prepared_dir / f"source_month_labels_{experiment}.csv"
catalog_path = prepared_dir / f"patch_catalog_{experiment}.csv"
stats_path = prepared_dir / f"channel_stats_{experiment}.csv"
known_source_exclusion_path = prepared_dir / f"known_source_exclusion_{experiment}.tif"
model_dir = project / "models" / experiment / RUN_ID
target_dir.mkdir(parents=True, exist_ok=True)
model_dir.mkdir(parents=True, exist_ok=True)
print("Unique model output directory:", model_dir)

XCH4_RE = re.compile(r"^XCH4_mean_(?P<year>20\d{2})_(?P<month>\d{2})(?:_3deg)?_CONUSbuf2p5deg\.tif$")
ANOM3_RE = re.compile(r"^anomaly_(?P<year>20\d{2})_(?P<month>\d{2})_3deg_CONUSbuf2p5deg\.tif$")
ENH3_RE = re.compile(r"^enhancement_(?P<year>20\d{2})_(?P<month>\d{2})_3deg_CONUSbuf2p5deg\.tif$")
ANOM5_RE = re.compile(r"^anomaly_(?P<year>20\d{2})_(?P<month>\d{2})_5deg_CONUSbuf2p5deg\.tif$")
ENH5_RE = re.compile(
    r"^enhancement(?:GE2)?_(?P<year>20\d{2})_(?P<month>\d{2})_5deg(?:_continuous)?_CONUSbuf2p5deg\.tif$"
)
BIVAR_RE = re.compile(
    r"^bivariate_(?:custombins|pixel)_(?P<year>20\d{2})_(?P<month>\d{2})_CONUS(?:_K4plus|_K4)?\.tif$"
)

CHANNEL_ROLES = ["xch4_mean", "anomaly_3deg", "enhancement_3deg", "anomaly_5deg", "enhancement_5deg"]
SCREENING_ROLES = ["bivariate"]
REQUIRED_RASTER_ROLES = CHANNEL_ROLES + SCREENING_ROLES


def split_for_month(year, month):
    ym = int(year) * 100 + int(month)
    if 202301 <= ym <= 202505:
        return "train"
    if ym in (202506, 202507, 202508):
        return "val"
    if ym in (202509, 202510, 202511, 202512):
        return "test"
    raise ValueError((year, month))


def raster_search_roots(project_dir):
    roots = []

    def add_root(path):
        path = Path(path)
        try:
            exists = path.exists()
        except OSError as exc:
            print(f"Skipping unavailable raster search root: {path} ({exc})")
            return
        if exists and path not in roots:
            roots.append(path)

    add_root(project_dir)
    for root in globals().get("EXTRA_RASTER_ROOTS", []):
        add_root(root)
    add_root("/content/drive/MyDrive/CNN_point_source")
    add_root("/content/drive/MyDrive/CNN_pixel_level")

    return roots


def register_path(rows, match, role, path):
    if not match:
        return
    year = int(match.group("year"))
    month = int(match.group("month"))
    if year in cfg.years:
        rows.setdefault((year, month), {}).setdefault(role, str(path))


def tif_sort_key(path):
    name = path.name
    prefer_continuous = 0 if "_continuous_" in name else 1
    prefer_not_ge2 = 0 if not name.startswith("enhancementGE2_") else 1
    return (prefer_continuous, prefer_not_ge2, str(path))


def find_monthly_manifest(project_dir):
    rows = {}
    roots = raster_search_roots(project_dir)
    print("Raster search roots:")
    for root in roots:
        print(" -", root)

    scanned = 0
    for root in roots:
        try:
            tif_paths = sorted(root.rglob("*.tif"), key=tif_sort_key)
        except OSError as exc:
            print(f"Skipping unavailable raster search root during scan: {root} ({exc})")
            continue

        for path in tif_paths:
            scanned += 1
            register_path(rows, XCH4_RE.match(path.name), "xch4_mean", path)
            register_path(rows, ANOM3_RE.match(path.name), "anomaly_3deg", path)
            register_path(rows, ENH3_RE.match(path.name), "enhancement_3deg", path)
            register_path(rows, ANOM5_RE.match(path.name), "anomaly_5deg", path)
            register_path(rows, ENH5_RE.match(path.name), "enhancement_5deg", path)
            register_path(rows, BIVAR_RE.match(path.name), "bivariate", path)

    records = []
    incomplete = []
    for (year, month), paths in sorted(rows.items()):
        missing_roles = [role for role in REQUIRED_RASTER_ROLES if role not in paths]
        if missing_roles:
            incomplete.append({
                "year": year,
                "month": month,
                "missing": ", ".join(missing_roles),
                "found": ", ".join(sorted(paths)),
            })
            continue

        records.append({
            "year": year,
            "month": month,
            "ym": f"{year}-{month:02d}",
            "split": split_for_month(year, month),
            "xch4_mean": paths["xch4_mean"],
            "anomaly_3deg": paths["anomaly_3deg"],
            "enhancement_3deg": paths["enhancement_3deg"],
            "anomaly_5deg": paths["anomaly_5deg"],
            "enhancement_5deg": paths["enhancement_5deg"],
            "bivariate": paths["bivariate"],
        })

    manifest = pd.DataFrame(records)
    if not manifest.empty:
        manifest = manifest.sort_values(["year", "month"]).reset_index(drop=True)

    expected = {(year, month) for year in cfg.years for month in range(1, 13)}
    found = set(zip(manifest["year"], manifest["month"])) if not manifest.empty else set()
    missing_months = sorted(expected - found)

    print("Total tif files scanned:", scanned)
    if incomplete:
        print("\nIncomplete month/channel combinations:")
        display(pd.DataFrame(incomplete).head(80))
    if missing_months:
        print("\nMissing complete required months:", missing_months)
    if manifest.empty:
        raise FileNotFoundError(
            "No complete XCH4 + anomaly_3deg + enhancement_3deg + anomaly_5deg + enhancement_5deg + bivariate month sets found."
        )
    return manifest


manifest = find_monthly_manifest(project)
reference_raster = manifest.iloc[0]["xch4_mean"]
with rasterio.open(reference_raster) as ref_src:
    ref_transform = ref_src.transform
    ref_crs = ref_src.crs
    ref_width = ref_src.width
    ref_height = ref_src.height
    ref_bounds = ref_src.bounds

print("Monthly multi-scale XCH4 + anomaly/enhancement + bivariate rasters found:")
print(manifest.groupby(["year", "split"]).size())
print("Reference grid:", ref_width, ref_height, ref_crs, ref_bounds)
print("CNN channels:", CHANNEL_ROLES)
print("Negative screening rasters:", SCREENING_ROLES)
display(manifest[["ym", "split", "xch4_mean", "anomaly_3deg", "enhancement_3deg", "anomaly_5deg", "enhancement_5deg", "bivariate"]].head())


## 4. Build Monthly Carbon Mapper Labels

Only plume-level information is used for activity. A source is active in a
month only if a CH4 plume is attributed to that source in that same month.

`persistence` and `detection_date_count` from the source table are not used
here because they summarize a broader source table time window.


In [ ]:
POINT_WKT_RE = re.compile(r"POINT\s*\(\s*(?P<lon>[-+0-9.eE]+)\s+(?P<lat>[-+0-9.eE]+)\s*\)")


def parse_point_wkt(value):
    if not isinstance(value, str):
        return None
    match = POINT_WKT_RE.search(value)
    if not match:
        return None
    return float(match.group("lon")), float(match.group("lat"))


def build_source_month_labels():
    plumes = pd.read_csv(plumes_csv)
    plumes["scene_timestamp"] = pd.to_datetime(
        plumes["scene_timestamp"],
        errors="coerce",
        utc=True,
        format="mixed",
    )
    plumes = plumes[
        plumes["gas"].astype(str).str.upper().eq("CH4")
        & plumes["scene_timestamp"].dt.year.isin(cfg.years)
        & plumes["point_source_name"].notna()
    ].copy()
    plumes["year"] = plumes["scene_timestamp"].dt.year.astype(int)
    plumes["month"] = plumes["scene_timestamp"].dt.month.astype(int)
    plumes["ym"] = plumes["scene_timestamp"].dt.strftime("%Y-%m")
    plumes["emission_auto"] = pd.to_numeric(plumes["emission_auto"], errors="coerce").fillna(0.0).clip(lower=0.0)

    source_table = pd.read_csv(sources_csv)
    source_table = source_table.rename(columns={"source_name": "point_source_name"})
    coords = source_table["geometry"].apply(parse_point_wkt)
    source_table["lon_source"] = coords.apply(lambda xy: xy[0] if xy else np.nan)
    source_table["lat_source"] = coords.apply(lambda xy: xy[1] if xy else np.nan)
    source_cols = ["point_source_name", "gas", "sector", "geometry", "lon_source", "lat_source"]
    source_table = source_table[source_cols].drop_duplicates("point_source_name")

    grouped = (
        plumes.groupby(["point_source_name", "year", "month", "ym"], as_index=False)
        .agg(
            plume_count=("plume_id", "count"),
            scene_count=("scene_id", "nunique"),
            detection_day_count=("scene_timestamp", lambda x: x.dt.date.nunique()),
            emission_sum=("emission_auto", "sum"),
            emission_mean=("emission_auto", "mean"),
            first_scene=("scene_timestamp", "min"),
            last_scene=("scene_timestamp", "max"),
            lon_plume=("plume_lon", "mean"),
            lat_plume=("plume_lat", "mean"),
        )
    )

    labels = grouped.merge(source_table, on="point_source_name", how="left")
    labels["lon"] = labels["lon_source"].fillna(labels["lon_plume"])
    labels["lat"] = labels["lat_source"].fillna(labels["lat_plume"])
    labels = labels.dropna(subset=["lon", "lat"]).reset_index(drop=True)

    available_months = set(zip(manifest["year"], manifest["month"]))
    labels = labels[labels[["year", "month"]].apply(tuple, axis=1).isin(available_months)].copy()
    labels["split"] = labels.apply(lambda r: split_for_month(int(r["year"]), int(r["month"])), axis=1)

    return labels.sort_values(["year", "month", "point_source_name"]).reset_index(drop=True)


if labels_path.exists() and not cfg.rebuild_labels:
    labels = pd.read_csv(labels_path)
else:
    labels = build_source_month_labels()
    labels.to_csv(labels_path, index=False)

print("Source-month labels:", len(labels))
print(labels.groupby(["year", "month", "split"]).size())
print("\nSplit label counts:")
print(labels["split"].value_counts())
print("\nUnique active source names:", labels["point_source_name"].nunique())


## 5. Create Monthly Source-Presence Target Rasters

The Gaussian target is only used to decide whether a patch contains a source.
The classifier target is still binary at patch level.


In [ ]:
def create_unweighted_gaussian_target(month_labels, reference_raster, output_tif, sigma_pixels=3.0, truncate=4.0):
    output_tif = Path(output_tif)
    output_tif.parent.mkdir(parents=True, exist_ok=True)
    with rasterio.open(reference_raster) as ref:
        profile = ref.profile.copy()
        transform = ref.transform
        height, width = ref.height, ref.width

    target = np.zeros((height, width), dtype=np.float32)
    radius = int(np.ceil(truncate * sigma_pixels))
    offsets = np.arange(-radius, radius + 1)
    yy, xx = np.meshgrid(offsets, offsets, indexing="ij")
    kernel = np.exp(-0.5 * ((xx / sigma_pixels) ** 2 + (yy / sigma_pixels) ** 2)).astype(np.float32)

    for _, src in month_labels.dropna(subset=["lon", "lat"]).iterrows():
        col_f, row_f = ~transform * (float(src["lon"]), float(src["lat"]))
        center_row = int(round(row_f))
        center_col = int(round(col_f))
        if center_row < 0 or center_row >= height or center_col < 0 or center_col >= width:
            continue

        r0 = max(0, center_row - radius)
        r1 = min(height, center_row + radius + 1)
        c0 = max(0, center_col - radius)
        c1 = min(width, center_col + radius + 1)
        kr0 = r0 - (center_row - radius)
        kr1 = kr0 + (r1 - r0)
        kc0 = c0 - (center_col - radius)
        kc1 = kc0 + (c1 - c0)
        target[r0:r1, c0:c1] = np.maximum(target[r0:r1, c0:c1], kernel[kr0:kr1, kc0:kc1])

    profile.update(count=1, dtype="float32", nodata=0.0, compress="deflate", predictor=2)
    with rasterio.open(output_tif, "w", **profile) as dst:
        dst.write(target, 1)
        dst.set_band_description(1, "gaussian_ch4_source_target")
    return output_tif


def create_known_source_exclusion_target(source_points, reference_raster, output_tif):
    return create_unweighted_gaussian_target(
        month_labels=source_points,
        reference_raster=reference_raster,
        output_tif=output_tif,
        sigma_pixels=cfg.gaussian_sigma_pixels,
    )


target_paths = {}
for row in manifest.itertuples(index=False):
    year = int(row.year)
    month = int(row.month)
    out = target_dir / f"target_active_ch4_{year}_{month:02d}_gaussian.tif"
    target_paths[(year, month)] = out
    if out.exists() and not cfg.rebuild_targets:
        continue

    month_labels = labels[labels["year"].eq(year) & labels["month"].eq(month)]
    create_unweighted_gaussian_target(
        month_labels=month_labels,
        reference_raster=row.xch4_mean,
        output_tif=out,
        sigma_pixels=cfg.gaussian_sigma_pixels,
    )

known_source_points = labels[["point_source_name", "lon", "lat"]].drop_duplicates("point_source_name")
if cfg.rebuild_targets or not known_source_exclusion_path.exists():
    create_known_source_exclusion_target(
        source_points=known_source_points,
        reference_raster=reference_raster,
        output_tif=known_source_exclusion_path,
    )

print("Monthly targets:", len(target_paths), "in", target_dir)
print("Known-source exclusion target:", known_source_exclusion_path)
print("Unique known CM source locations excluded from negatives:", len(known_source_points))


## 6. Build Binary Patch Catalog With Center Class-1 Bivariate Negatives


In [ ]:
def non_overlapping_centers(height, width, patch_size):
    half = patch_size // 2
    for row0 in range(0, height - patch_size + 1, patch_size):
        row = row0 + half
        for col0 in range(0, width - patch_size + 1, patch_size):
            col = col0 + half
            yield row, col


def patch_window(row, col, patch_size):
    half = patch_size // 2
    return Window(col - half, row - half, patch_size, patch_size)


def patch_from_array(arr, row, col, patch_size):
    half = patch_size // 2
    return arr[row - half: row + half, col - half: col + half]


def patch_touches_target(target_arr, row, col, patch_size):
    patch = patch_from_array(target_arr, row, col, patch_size)
    return float(np.nanmax(patch)) > cfg.negative_target_max


def load_month_arrays(raster_row):
    arrays = {}
    for role in CHANNEL_ROLES:
        with rasterio.open(getattr(raster_row, role)) as src:
            arrays[role] = src.read(1, masked=True).filled(np.nan).astype(np.float32)
    for role in ["enhancement_3deg", "enhancement_5deg"]:
        enhancement_path = Path(getattr(raster_row, role))
        arrays[f"_{role}_is_ge2_mask"] = enhancement_path.name.startswith("enhancementGE2_")
    return arrays


def enhancement_role_patch_stats(arrays, role, row, col, patch_size):
    patch = patch_from_array(arrays[role], row, col, patch_size)
    if not np.any(np.isfinite(patch)):
        return np.nan, np.nan, np.nan
    valid = patch[np.isfinite(patch)]
    is_ge2_mask = bool(arrays.get(f"_{role}_is_ge2_mask", False))
    ge2_fraction = float(np.mean(valid > 0)) if is_ge2_mask else float(np.mean(valid >= cfg.enhancement_area_threshold))
    return (
        float(np.nanmax(valid)),
        float(np.mean(valid > 0)),
        ge2_fraction,
    )


def enhancement_patch_stats(arrays, row, col, patch_size):
    stats = [
        enhancement_role_patch_stats(arrays, role, row, col, patch_size)
        for role in ["enhancement_3deg", "enhancement_5deg"]
    ]
    max_values = [value[0] for value in stats if np.isfinite(value[0])]
    positive_fractions = [value[1] for value in stats if np.isfinite(value[1])]
    ge2_fractions = [value[2] for value in stats if np.isfinite(value[2])]
    if not max_values:
        return np.nan, np.nan, np.nan
    return (
        float(np.max(max_values)),
        float(np.max(positive_fractions)),
        float(np.max(ge2_fractions)),
    )


def bivariate_center_class(bivar_src, target_transform, row, col):
    lon, lat = target_transform * (float(col), float(row))
    bivar_row, bivar_col = bivar_src.index(lon, lat)

    if (
        bivar_row < 0
        or bivar_row >= bivar_src.height
        or bivar_col < 0
        or bivar_col >= bivar_src.width
    ):
        return None

    value = bivar_src.read(1, window=Window(bivar_col, bivar_row, 1, 1))[0, 0]
    if bivar_src.nodata is not None and value == bivar_src.nodata:
        return None
    for nodata_value in cfg.bivariate_nodata_values:
        if value == nodata_value:
            return None
    return int(value)


def shifted_center(row, col, dy, dx, patch_size, height, width):
    half = patch_size // 2
    row = int(np.clip(int(row) + int(dy), half, height - half - 1))
    col = int(np.clip(int(col) + int(dx), half, width - half - 1))
    return row, col


def check_screening_grid(target_src, bivar_src, ym):
    if target_src.crs != bivar_src.crs:
        raise RuntimeError(
            f"{ym}: bivariate CRS does not match target CRS. "
            "This notebook does not resample bivariate rasters; provide a matching CRS raster."
        )


def build_patch_catalog():
    rng = np.random.default_rng(cfg.seed)
    rows = []

    with rasterio.open(known_source_exclusion_path) as known_src:
        known_arr = known_src.read(1, masked=True).filled(0.0).astype(np.float32)

    for raster_row in tqdm(list(manifest.itertuples(index=False)), desc="non-overlap catalog"):
        year = int(raster_row.year)
        month = int(raster_row.month)
        ym = f"{year}-{month:02d}"
        split = raster_row.split
        target_path = target_paths[(year, month)]

        arrays = load_month_arrays(raster_row)

        positives = []
        negative_candidates = []
        rejected_known_source = 0
        rejected_bivariate_center = 0
        rejected_enhancement_area = 0

        with rasterio.open(target_path) as target_src, rasterio.open(raster_row.bivariate) as bivar_src:
            check_screening_grid(target_src, bivar_src, ym)
            target_arr = target_src.read(1, masked=True).filled(0.0).astype(np.float32)
            height = target_src.height
            width = target_src.width
            target_transform = target_src.transform

            for row, col in non_overlapping_centers(height, width, cfg.patch_size):
                touches_monthly_source = patch_touches_target(target_arr, row, col, cfg.patch_size)

                if touches_monthly_source:
                    positives.append(
                        {
                            "year": year,
                            "month": month,
                            "ym": ym,
                            "row": int(row),
                            "col": int(col),
                            "split": split,
                            "label": 1,
                            "sample_type": "positive_nonoverlap",
                            "enhancement_patch_max": np.nan,
                            "enhancement_positive_fraction": np.nan,
                            "enhancement_ge2_fraction": np.nan,
                            "bivariate_center_class": np.nan,
                        }
                    )
                    continue

                # Negative rule 1 and 2:
                # no active source this month and no known CM source from any
                # 2023-2025 month in the patch.
                if patch_touches_target(known_arr, row, col, cfg.patch_size):
                    rejected_known_source += 1
                    continue

                # Negative rule 3: the bivariate raster center pixel must be
                # class 1, i.e. low-exceedance/background.
                center_class = bivariate_center_class(bivar_src, target_transform, row, col)
                if center_class != cfg.bivariate_center_class:
                    rejected_bivariate_center += 1
                    continue

                # Final negative filter: reject only meaningful enhancement
                # areas, not tiny/noisy positive values.
                enh_max, enh_positive_fraction, enh_ge2_fraction = enhancement_patch_stats(
                    arrays,
                    row,
                    col,
                    cfg.patch_size,
                )
                if enh_ge2_fraction > cfg.max_enhancement_area_fraction:
                    rejected_enhancement_area += 1
                    continue

                negative_candidates.append(
                    {
                        "year": year,
                        "month": month,
                        "ym": ym,
                        "row": int(row),
                        "col": int(col),
                        "split": split,
                        "label": 0,
                        "sample_type": "strict_negative_bivar1_nonoverlap",
                        "enhancement_patch_max": enh_max,
                        "enhancement_positive_fraction": enh_positive_fraction,
                        "enhancement_ge2_fraction": enh_ge2_fraction,
                        "bivariate_center_class": center_class,
                    }
                )

        chosen_negatives = negative_candidates

        rows.extend(positives)
        rows.extend(chosen_negatives)

        if split == "train" and cfg.shift_augment_training:
            used_centers = {
                (int(item["row"]), int(item["col"]))
                for item in positives + chosen_negatives
            }

            augmented = []

            with rasterio.open(target_path) as target_src, rasterio.open(raster_row.bivariate) as bivar_src:
                check_screening_grid(target_src, bivar_src, ym)
                target_arr = target_src.read(1, masked=True).filled(0.0).astype(np.float32)
                height = target_src.height
                width = target_src.width
                target_transform = target_src.transform

                for base in positives + chosen_negatives:
                    made = 0
                    tries = 0
                    while (
                        made < cfg.train_shift_augmentations_per_patch
                        and tries < cfg.train_shift_max_tries_per_patch
                    ):
                        tries += 1
                        dy = int(rng.integers(-cfg.train_shift_max_pixels, cfg.train_shift_max_pixels + 1))
                        dx = int(rng.integers(-cfg.train_shift_max_pixels, cfg.train_shift_max_pixels + 1))
                        if dy == 0 and dx == 0:
                            continue

                        aug_row, aug_col = shifted_center(
                            base["row"],
                            base["col"],
                            dy,
                            dx,
                            cfg.patch_size,
                            height,
                            width,
                        )
                        center_key = (aug_row, aug_col)
                        if center_key in used_centers:
                            continue

                        if int(base["label"]) == 1:
                            if not patch_touches_target(target_arr, aug_row, aug_col, cfg.patch_size):
                                continue

                            augmented.append(
                                {
                                    **base,
                                    "row": int(aug_row),
                                    "col": int(aug_col),
                                    "sample_type": "positive_shift_augmented",
                                    "base_row": int(base["row"]),
                                    "base_col": int(base["col"]),
                                    "shift_dy": int(dy),
                                    "shift_dx": int(dx),
                                }
                            )
                        else:
                            if patch_touches_target(target_arr, aug_row, aug_col, cfg.patch_size):
                                continue
                            if patch_touches_target(known_arr, aug_row, aug_col, cfg.patch_size):
                                continue

                            center_class = bivariate_center_class(
                                bivar_src,
                                target_transform,
                                aug_row,
                                aug_col,
                            )
                            if center_class != cfg.bivariate_center_class:
                                continue

                            enh_max, enh_positive_fraction, enh_ge2_fraction = enhancement_patch_stats(
                                arrays,
                                aug_row,
                                aug_col,
                                cfg.patch_size,
                            )
                            if enh_ge2_fraction > cfg.max_enhancement_area_fraction:
                                continue

                            augmented.append(
                                {
                                    **base,
                                    "row": int(aug_row),
                                    "col": int(aug_col),
                                    "sample_type": "strict_negative_shift_augmented",
                                    "enhancement_patch_max": enh_max,
                                    "enhancement_positive_fraction": enh_positive_fraction,
                                    "enhancement_ge2_fraction": enh_ge2_fraction,
                                    "bivariate_center_class": center_class,
                                    "base_row": int(base["row"]),
                                    "base_col": int(base["col"]),
                                    "shift_dy": int(dy),
                                    "shift_dx": int(dx),
                                }
                            )

                        used_centers.add(center_key)
                        made += 1

            rows.extend(augmented)
            print(
                f"{ym} {split}: shift_augmented={len(augmented)}, "
                f"shift_augmented_positive={sum(int(x['label']) == 1 for x in augmented)}, "
                f"shift_augmented_negative={sum(int(x['label']) == 0 for x in augmented)}"
            )

        print(
            f"{ym} {split}: positives={len(positives)}, "
            f"strict_negative_candidates={len(negative_candidates)}, "
            f"kept_strict_negatives={len(chosen_negatives)}, "
            f"rejected_known_source={rejected_known_source}, "
            f"rejected_bivariate_center={rejected_bivariate_center}, "
            f"rejected_enhancement_area={rejected_enhancement_area}"
        )

    catalog = pd.DataFrame(rows)
    if catalog.empty:
        raise RuntimeError("Patch catalog is empty. Check labels, targets, bivariate rasters, and strict negative rules.")

    catalog = catalog.sample(frac=1.0, random_state=cfg.seed).reset_index(drop=True)
    return catalog


if catalog_path.exists() and not cfg.rebuild_patch_catalog:
    catalog = pd.read_csv(catalog_path)
else:
    catalog = build_patch_catalog()
    catalog.to_csv(catalog_path, index=False)

print(catalog.groupby(["split", "label"]).size().unstack(fill_value=0))
print("\nBy sample type:")
print(catalog.groupby(["split", "sample_type"]).size().unstack(fill_value=0))
print("\nStrict negative diagnostics:")
negative_diag_cols = [
    "enhancement_patch_max",
    "enhancement_positive_fraction",
    "enhancement_ge2_fraction",
    "bivariate_center_class",
]
if set(negative_diag_cols).issubset(catalog.columns):
    print(catalog[catalog["label"].eq(0)][negative_diag_cols].describe())
print("\nBy month:")
print(catalog.groupby(["ym", "split", "label"]).size().unstack(fill_value=0).fillna(0).astype(int))
print("Catalog:", catalog_path, len(catalog))


## 7. Raster Patch Reader And Normalization


In [ ]:
class MonthlyPatchReader:
    def __init__(self, manifest_df, patch_size):
        self.manifest = manifest_df.copy()
        self.patch_size = patch_size
        self.sources = {}

    def open(self):
        if not self.sources:
            for row in self.manifest.itertuples(index=False):
                for role in CHANNEL_ROLES:
                    self.sources[(int(row.year), int(row.month), role)] = rasterio.open(getattr(row, role))
        return self

    def close(self):
        for src in self.sources.values():
            src.close()
        self.sources = {}

    def read(self, year, month, row, col):
        self.open()
        half = self.patch_size // 2
        window = Window(col - half, row - half, self.patch_size, self.patch_size)
        arrays = []

        for role in CHANNEL_ROLES:
            src = self.sources[(int(year), int(month), role)]
            arr = src.read(
                1,
                window=window,
                out_shape=(self.patch_size, self.patch_size),
                boundless=True,
                fill_value=np.nan,
                resampling=Resampling.bilinear,
            ).astype(np.float32)
            arrays.append(arr)

        return np.stack(arrays, axis=0)


reader = MonthlyPatchReader(manifest, cfg.patch_size)
x0 = reader.read(int(catalog.iloc[0].year), int(catalog.iloc[0].month), int(catalog.iloc[0].row), int(catalog.iloc[0].col))
reader.close()
print(x0.shape, np.nanmin(x0), np.nanmax(x0), "label:", catalog.iloc[0].label)


def compute_channel_stats():
    if stats_path.exists():
        stats = pd.read_csv(stats_path)
        return stats["mean"].to_numpy(np.float32), stats["std"].to_numpy(np.float32)

    train_catalog = catalog[catalog["split"].eq("train")].sample(
        n=min(cfg.stats_windows, catalog["split"].eq("train").sum()),
        random_state=cfg.seed,
    )

    n_channels = len(CHANNEL_ROLES)
    count = np.zeros(n_channels, dtype=np.float64)
    sums = np.zeros(n_channels, dtype=np.float64)
    sums2 = np.zeros(n_channels, dtype=np.float64)

    r = MonthlyPatchReader(manifest, cfg.patch_size)
    for item in tqdm(train_catalog.itertuples(index=False), total=len(train_catalog), desc="stats"):
        x = r.read(int(item.year), int(item.month), int(item.row), int(item.col))
        valid = np.isfinite(x)
        x0 = np.where(valid, x, 0.0)
        count += valid.reshape(n_channels, -1).sum(axis=1)
        sums += x0.reshape(n_channels, -1).sum(axis=1)
        sums2 += (x0 * x0).reshape(n_channels, -1).sum(axis=1)
    r.close()

    mean = sums / np.maximum(count, 1)
    var = sums2 / np.maximum(count, 1) - mean * mean
    std = np.sqrt(np.maximum(var, 1e-6))
    stats = pd.DataFrame({"channel": CHANNEL_ROLES, "mean": mean, "std": std})
    stats.to_csv(stats_path, index=False)
    return mean.astype(np.float32), std.astype(np.float32)


channel_mean, channel_std = compute_channel_stats()
print("channel_mean:", channel_mean)
print("channel_std:", channel_std)


## 8. Dataset And DataLoaders


In [ ]:
N_PHYS = 0
PHYS_FEATURE_NAMES = []


class MonthlyPatchDataset(Dataset):
    def __init__(self, catalog_df, manifest_df, mean, std, patch_size, augment=False):
        self.catalog = catalog_df.reset_index(drop=True)
        self.reader = MonthlyPatchReader(manifest_df, patch_size)
        self.mean = np.asarray(mean, dtype=np.float32)[:, None, None]
        self.std = np.asarray(std, dtype=np.float32)[:, None, None]
        self.augment = augment

    def __len__(self):
        return len(self.catalog)

    def __getitem__(self, idx):
        item = self.catalog.iloc[idx]
        x_raw = self.reader.read(
            int(item.year),
            int(item.month),
            int(item.row),
            int(item.col),
        ).astype(np.float32)
        x = (x_raw - self.mean) / self.std
        x = np.nan_to_num(
            x,
            nan=0.0,
            posinf=0.0,
            neginf=0.0,
        ).astype(np.float32)
        phys = np.zeros((N_PHYS,), dtype=np.float32)
        y = np.array([float(item.label)], dtype=np.float32)
        return torch.from_numpy(x), torch.from_numpy(phys), torch.from_numpy(y)


train_ds = MonthlyPatchDataset(
    catalog[catalog.split == "train"],
    manifest,
    channel_mean,
    channel_std,
    cfg.patch_size,
    augment=False,
)
val_ds = MonthlyPatchDataset(
    catalog[catalog.split == "val"],
    manifest,
    channel_mean,
    channel_std,
    cfg.patch_size,
    augment=False,
)
test_ds = MonthlyPatchDataset(
    catalog[catalog.split == "test"],
    manifest,
    channel_mean,
    channel_std,
    cfg.patch_size,
    augment=False,
)

# Natural training: no replacement sampler and no negative removal.
# Every catalog row is visited exactly once in each epoch, in shuffled order.
train_loader = DataLoader(
    train_ds,
    batch_size=cfg.batch_size,
    shuffle=True,
    num_workers=cfg.num_workers,
    pin_memory=True,
)
val_loader = DataLoader(
    val_ds,
    batch_size=cfg.batch_size,
    shuffle=False,
    num_workers=cfg.num_workers,
    pin_memory=True,
)
test_loader = DataLoader(
    test_ds,
    batch_size=cfg.batch_size,
    shuffle=False,
    num_workers=cfg.num_workers,
    pin_memory=True,
)

train_counts_for_loader = train_ds.catalog["label"].value_counts().sort_index()
print("Dataset sizes:", len(train_ds), len(val_ds), len(test_ds))
print("Natural train label counts:")
print(train_counts_for_loader)
print("Balanced/replacement sampler enabled:", False)
print("Every natural training sample used once per epoch:", True)
print("Online flip/rotation augmentation enabled:", False)
print("Catalog shift augmentation enabled:", cfg.shift_augment_training)
xb, pb, yb = next(iter(train_loader))
print("Raster batch:", xb.shape)
print("Physical feature batch:", pb.shape)
print("Label batch:", yb.shape)
print("First shuffled batch positives:", int(yb.sum().item()), "/", len(yb))


## 9. CNN Classifier


In [ ]:
class SmallCNNBranch(nn.Module):
    def __init__(self, in_channels=1, base=16, dropout=0.15):
        super().__init__()
        self.net = nn.Sequential(
            nn.Conv2d(in_channels, base, kernel_size=3, padding=1, bias=False),
            nn.BatchNorm2d(base),
            nn.SiLU(inplace=True),
            nn.Conv2d(base, base, kernel_size=3, padding=1, bias=False),
            nn.BatchNorm2d(base),
            nn.SiLU(inplace=True),
            nn.MaxPool2d(2),
            nn.Conv2d(base, base * 2, kernel_size=3, padding=1, bias=False),
            nn.BatchNorm2d(base * 2),
            nn.SiLU(inplace=True),
            nn.Conv2d(base * 2, base * 2, kernel_size=3, padding=1, bias=False),
            nn.BatchNorm2d(base * 2),
            nn.SiLU(inplace=True),
            nn.Dropout2d(dropout),
            nn.MaxPool2d(2),
            nn.Conv2d(base * 2, base * 4, kernel_size=3, padding=1, bias=False),
            nn.BatchNorm2d(base * 4),
            nn.SiLU(inplace=True),
            nn.AdaptiveAvgPool2d(1),
            nn.Flatten(),
        )

    def forward(self, x):
        return self.net(x)


class MultiScaleMethaneNet(nn.Module):
    def __init__(self, n_channels, base=16, dropout=0.6):
        super().__init__()
        self.branches = nn.ModuleList([
            SmallCNNBranch(in_channels=1, base=base, dropout=0.15)
            for _ in range(n_channels)
        ])

        branch_dim = base * 4
        merged_dim = branch_dim * n_channels
        self.classifier = nn.Sequential(
            nn.Dropout(dropout),
            nn.Linear(merged_dim, base * 4),
            nn.SiLU(inplace=True),
            nn.Dropout(dropout),
            nn.Linear(base * 4, 1),
        )

    def forward(self, x, phys=None):
        features = [
            branch(x[:, i:i + 1, :, :])
            for i, branch in enumerate(self.branches)
        ]
        merged = torch.cat(features, dim=1)
        return self.classifier(merged)


N_PHYS = 0
PHYS_FEATURE_NAMES = []

model = MultiScaleMethaneNet(
    n_channels=len(CHANNEL_ROLES),
    base=cfg.base_channels,
    dropout=cfg.dropout,
).to(cfg.device)

print("Channels:", CHANNEL_ROLES)
print("Physical features removed:", N_PHYS == 0)
print(sum(p.numel() for p in model.parameters()) / 1e6, "M parameters")


Optional architecture summary.


In [ ]:
try:
    from torchinfo import summary
except ModuleNotFoundError:
    !pip -q install torchinfo
    from torchinfo import summary

if N_PHYS > 0:
    summary_input_size = [
        (cfg.batch_size, len(CHANNEL_ROLES), cfg.patch_size, cfg.patch_size),
        (cfg.batch_size, N_PHYS),
    ]
else:
    summary_input_size = (
        cfg.batch_size,
        len(CHANNEL_ROLES),
        cfg.patch_size,
        cfg.patch_size,
    )

summary(
    model,
    input_size=summary_input_size,
    col_names=("input_size", "output_size", "num_params", "trainable"),
    depth=4,
    device=cfg.device,
)


## 10. Natural-Distribution Training With Three Validation Checkpoints


In [ ]:
train_counts = catalog[catalog.split == "train"]["label"].value_counts()
neg = float(train_counts.get(0, 1))
pos = float(train_counts.get(1, 1))
pos_weight = torch.tensor([cfg.pos_weight], dtype=torch.float32, device=cfg.device)

print("train negatives:", neg)
print("train positives:", pos)
print("natural train negative:positive ratio:", neg / max(pos, 1.0))
print("training uses every sample once per epoch:", True)
print("training pos_weight:", float(pos_weight.item()))
print("validation/test loss uses pos_weight:", 1.0)

# Weight positives moderately during optimization. Validation and test loss are
# deliberately unweighted so model selection reflects their natural prevalence.
train_criterion = torch.nn.BCEWithLogitsLoss(pos_weight=pos_weight)
eval_criterion = torch.nn.BCEWithLogitsLoss()


def smooth_binary_targets(y, smoothing):
    return y * (1.0 - smoothing) + 0.5 * smoothing


def prepare_targets(targets):
    targets = targets.float()
    if targets.ndim == 1:
        targets = targets.view(-1, 1)
    return smooth_binary_targets(targets, cfg.label_smoothing)


def train_loss_fn(logits, targets):
    return train_criterion(logits, prepare_targets(targets))


def eval_loss_fn(logits, targets):
    return eval_criterion(logits, prepare_targets(targets))


@torch.no_grad()
def predict_loader(loader):
    model.eval()
    y_true = []
    y_prob = []
    losses = []
    seen = 0

    for x, phys, y in loader:
        x = x.to(cfg.device, non_blocking=True)
        phys = phys.to(cfg.device, non_blocking=True).float()
        y = y.to(cfg.device, non_blocking=True).float()

        if y.ndim == 1:
            y = y.view(-1, 1)

        logits = model(x, phys)
        loss = eval_loss_fn(logits, y)
        prob = torch.sigmoid(logits)

        batch_size = x.size(0)
        losses.append(loss.item() * batch_size)
        seen += batch_size
        y_true.append(y.detach().cpu().numpy().ravel())
        y_prob.append(prob.detach().cpu().numpy().ravel())

    y_true = np.concatenate(y_true) if y_true else np.array([])
    y_prob = np.concatenate(y_prob) if y_prob else np.array([])

    return y_true, y_prob, sum(losses) / max(seen, 1)


def classification_metrics(y_true, y_prob, threshold=0.5):
    if len(y_true) == 0:
        return {
            "accuracy": np.nan,
            "precision": np.nan,
            "recall": np.nan,
            "f1": np.nan,
            "auroc": np.nan,
            "auprc": np.nan,
        }

    y_pred = (y_prob >= threshold).astype(int)
    has_both_classes = len(np.unique(y_true)) > 1

    return {
        "accuracy": accuracy_score(y_true, y_pred),
        "precision": precision_score(y_true, y_pred, zero_division=0),
        "recall": recall_score(y_true, y_pred, zero_division=0),
        "f1": f1_score(y_true, y_pred, zero_division=0),
        "auroc": roc_auc_score(y_true, y_prob) if has_both_classes else np.nan,
        "auprc": average_precision_score(y_true, y_prob) if has_both_classes else np.nan,
    }


def find_best_threshold(y_true, y_prob):
    thresholds = np.linspace(0.01, 0.99, 99)
    best_t = 0.5
    best_f1 = -1.0

    for threshold in thresholds:
        y_pred = (y_prob >= threshold).astype(int)
        score = f1_score(y_true, y_pred, zero_division=0)
        if score > best_f1:
            best_f1 = score
            best_t = float(threshold)

    return best_t, best_f1


def safe_metric_value(value, fallback=-float("inf")):
    value = float(value)
    if not np.isfinite(value):
        return fallback
    return value


def checkpoint_payload(
    epoch,
    selection_metric,
    val_loss,
    val_auprc,
    val_auroc,
    best_threshold,
    extra=None,
):
    payload = {
        "model_state": model.state_dict(),
        "cfg": cfg.__dict__,
        "run_id": RUN_ID,
        "channel_mean": channel_mean,
        "channel_std": channel_std,
        "channel_roles": CHANNEL_ROLES,
        "n_phys": int(N_PHYS),
        "manifest": manifest.to_dict("records"),
        "pos_weight": float(pos_weight.item()),
        "sampling": "natural_shuffle_all_samples_once_per_epoch",
        "selection_metric": selection_metric,
        "best_val_auprc": float(val_auprc),
        "best_val_auroc": float(val_auroc),
        "best_val_loss": float(val_loss),
        "best_epoch": int(epoch),
        "best_threshold": float(best_threshold),
        "label_smoothing": float(cfg.label_smoothing),
    }
    if extra:
        payload.update(extra)
    return payload


optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=cfg.lr,
    weight_decay=cfg.weight_decay,
)

scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
    optimizer,
    mode="max",
    factor=0.5,
    patience=2,
)

use_amp = str(cfg.device).startswith("cuda")
scaler = torch.amp.GradScaler("cuda", enabled=use_amp)

best_selection_score = -float("inf")
best_epoch = 0
best_val_auprc = -float("inf")
best_val_auroc = -float("inf")
best_val_loss = float("inf")

best_val_recall_t05 = -float("inf")
best_recall_epoch = 0
best_recall_val_precision_t05 = -float("inf")
best_recall_val_f1_t05 = -float("inf")

best_val_f1_calibrated = -float("inf")
best_f1_epoch = 0
best_f1_threshold = 0.5
best_f1_val_precision = -float("inf")
best_f1_val_recall = -float("inf")

epochs_without_improvement = 0
history = []

selection_metric_name = "val_auprc_minus_0.05_val_loss"
recall_metric_name = "val_recall_t05"
f1_metric_name = "val_f1_best_threshold"

for epoch in range(1, cfg.epochs + 1):
    model.train()
    running = 0.0
    seen = 0

    pbar = tqdm(train_loader, desc=f"epoch {epoch}/{cfg.epochs}")

    for x, phys, y in pbar:
        x = x.to(cfg.device, non_blocking=True)
        phys = phys.to(cfg.device, non_blocking=True).float()
        y = y.to(cfg.device, non_blocking=True).float()

        if y.ndim == 1:
            y = y.view(-1, 1)

        optimizer.zero_grad(set_to_none=True)

        with torch.amp.autocast("cuda", enabled=use_amp):
            logits = model(x, phys)
            loss = train_loss_fn(logits, y)

        scaler.scale(loss).backward()
        scaler.unscale_(optimizer)
        torch.nn.utils.clip_grad_norm_(
            model.parameters(),
            max_norm=cfg.grad_clip_norm,
        )
        scaler.step(optimizer)
        scaler.update()

        batch_size = x.size(0)
        running += loss.item() * batch_size
        seen += batch_size
        pbar.set_postfix(train_loss=running / max(seen, 1))

    train_loss = running / max(seen, 1)

    y_val, p_val, val_loss = predict_loader(val_loader)
    val_metrics_05 = classification_metrics(y_val, p_val, threshold=0.5)
    best_threshold, _ = find_best_threshold(y_val, p_val)
    val_metrics_best_t = classification_metrics(
        y_val,
        p_val,
        threshold=best_threshold,
    )

    current_auroc = safe_metric_value(val_metrics_05["auroc"])
    current_auprc = safe_metric_value(val_metrics_05["auprc"])
    current_recall_t05 = safe_metric_value(val_metrics_05["recall"])
    current_f1_calibrated = safe_metric_value(val_metrics_best_t["f1"])

    selection_score = current_auprc - 0.05 * float(val_loss)

    improved = selection_score > (
        best_selection_score + cfg.early_stopping_min_delta
    )
    recall_improved = current_recall_t05 > (
        best_val_recall_t05 + cfg.early_stopping_min_delta
    )
    f1_improved = current_f1_calibrated > (
        best_val_f1_calibrated + cfg.early_stopping_min_delta
    )

    row = {
        "epoch": epoch,
        "train_loss_weighted": train_loss,
        "val_loss_unweighted": val_loss,
        "val_accuracy_t05": val_metrics_05["accuracy"],
        "val_precision_t05": val_metrics_05["precision"],
        "val_recall_t05": val_metrics_05["recall"],
        "val_f1_t05": val_metrics_05["f1"],
        "val_best_threshold": best_threshold,
        "val_accuracy_best_t": val_metrics_best_t["accuracy"],
        "val_precision_best_t": val_metrics_best_t["precision"],
        "val_recall_best_t": val_metrics_best_t["recall"],
        "val_f1_best_t": val_metrics_best_t["f1"],
        "val_auroc": current_auroc,
        "val_auprc": current_auprc,
        "selection_metric": selection_metric_name,
        "selection_score": selection_score,
        "recall_improved": recall_improved,
        "f1_improved": f1_improved,
        "lr": optimizer.param_groups[0]["lr"],
        "improved": improved,
    }

    history.append(row)
    print(row)
    scheduler.step(selection_score)

    if improved:
        best_selection_score = selection_score
        best_epoch = epoch
        best_val_auprc = current_auprc
        best_val_auroc = current_auroc
        best_val_loss = float(val_loss)
        epochs_without_improvement = 0

        torch.save(
            checkpoint_payload(
                epoch=epoch,
                selection_metric=selection_metric_name,
                val_loss=val_loss,
                val_auprc=current_auprc,
                val_auroc=current_auroc,
                best_threshold=best_threshold,
                extra={
                    "best_selection_score": float(best_selection_score),
                    "best_val_f1_calibrated": float(
                        val_metrics_best_t["f1"]
                    ),
                },
            ),
            model_dir / "patch_cnn_best.pt",
        )

        print(
            f"saved best-selection model at epoch {epoch} "
            f"with selection_score={best_selection_score:.4f}, "
            f"val_auprc={best_val_auprc:.4f}, "
            f"val_loss_unweighted={best_val_loss:.4f}, "
            f"validation_threshold={best_threshold:.2f}"
        )
    else:
        epochs_without_improvement += 1
        print(
            f"no validation selection-score improvement "
            f"for {epochs_without_improvement}/"
            f"{cfg.early_stopping_patience} epochs"
        )

    if recall_improved:
        best_val_recall_t05 = current_recall_t05
        best_recall_epoch = epoch
        best_recall_val_precision_t05 = safe_metric_value(
            val_metrics_05["precision"]
        )
        best_recall_val_f1_t05 = safe_metric_value(val_metrics_05["f1"])

        torch.save(
            checkpoint_payload(
                epoch=epoch,
                selection_metric=recall_metric_name,
                val_loss=val_loss,
                val_auprc=current_auprc,
                val_auroc=current_auroc,
                best_threshold=best_threshold,
                extra={
                    "best_val_recall_t05": float(best_val_recall_t05),
                    "best_val_precision_t05": float(
                        best_recall_val_precision_t05
                    ),
                    "best_val_f1_t05": float(best_recall_val_f1_t05),
                    "decision_threshold": 0.5,
                },
            ),
            model_dir / "patch_cnn_best_recall_t05.pt",
        )

        print(
            f"saved best-recall-t05 model at epoch {epoch} "
            f"with recall={best_val_recall_t05:.4f}, "
            f"precision={best_recall_val_precision_t05:.4f}, "
            f"f1={best_recall_val_f1_t05:.4f}"
        )

    if f1_improved:
        best_val_f1_calibrated = current_f1_calibrated
        best_f1_epoch = epoch
        best_f1_threshold = best_threshold
        best_f1_val_precision = safe_metric_value(
            val_metrics_best_t["precision"]
        )
        best_f1_val_recall = safe_metric_value(
            val_metrics_best_t["recall"]
        )

        torch.save(
            checkpoint_payload(
                epoch=epoch,
                selection_metric=f1_metric_name,
                val_loss=val_loss,
                val_auprc=current_auprc,
                val_auroc=current_auroc,
                best_threshold=best_threshold,
                extra={
                    "best_val_f1_calibrated": float(
                        best_val_f1_calibrated
                    ),
                    "best_val_precision_calibrated": float(
                        best_f1_val_precision
                    ),
                    "best_val_recall_calibrated": float(
                        best_f1_val_recall
                    ),
                },
            ),
            model_dir / "patch_cnn_best_f1_calibrated.pt",
        )

        print(
            f"saved best-calibrated-F1 model at epoch {epoch} "
            f"with val_f1={best_val_f1_calibrated:.4f}, "
            f"precision={best_f1_val_precision:.4f}, "
            f"recall={best_f1_val_recall:.4f}, "
            f"threshold={best_f1_threshold:.2f}"
        )

    if epochs_without_improvement >= cfg.early_stopping_patience:
        print(
            f"early stopping at epoch {epoch}; "
            f"best selection epoch {best_epoch}, "
            f"score {best_selection_score:.4f}; "
            f"best recall-t05 epoch {best_recall_epoch}, "
            f"recall {best_val_recall_t05:.4f}; "
            f"best calibrated-F1 epoch {best_f1_epoch}, "
            f"F1 {best_val_f1_calibrated:.4f}, "
            f"threshold {best_f1_threshold:.2f}"
        )
        break

history_df = pd.DataFrame(history)
history_df.to_csv(model_dir / "training_history.csv", index=False)
print("saved training history:", model_dir / "training_history.csv")


In [ ]:
# Plot validation AUPRC, AUROC, calibrated F1, and unweighted loss.

if "history_df" in globals():
    plot_df = history_df.copy()
elif "history" in globals():
    plot_df = pd.DataFrame(history).copy()
else:
    raise RuntimeError(
        "No history_df or history found. Run the training cell first."
    )

required_columns = [
    "epoch",
    "val_auprc",
    "val_auroc",
    "val_f1_best_t",
    "val_loss_unweighted",
]
missing = [column for column in required_columns if column not in plot_df]
if missing:
    print("Available columns:")
    print(list(plot_df.columns))
    raise RuntimeError(f"Missing required metric columns: {missing}")

plot_df = plot_df.sort_values("epoch")

fig, ax1 = plt.subplots(figsize=(10, 5))
ax1.plot(
    plot_df["epoch"],
    plot_df["val_auprc"],
    marker="o",
    label="Val AUPRC",
    color="tab:blue",
)
ax1.plot(
    plot_df["epoch"],
    plot_df["val_auroc"],
    marker="o",
    label="Val AUROC",
    color="tab:green",
)
ax1.plot(
    plot_df["epoch"],
    plot_df["val_f1_best_t"],
    marker="o",
    label="Val F1 at validation-selected threshold",
    color="tab:purple",
)
ax1.set_xlabel("Epoch")
ax1.set_ylabel("Validation metric")
ax1.set_ylim(0, 1)
ax1.grid(True, alpha=0.3)

ax2 = ax1.twinx()
ax2.plot(
    plot_df["epoch"],
    plot_df["val_loss_unweighted"],
    marker="o",
    label="Val BCE loss (unweighted)",
    color="tab:red",
    linestyle="--",
)
ax2.set_ylabel("Unweighted validation BCE loss")

lines_1, labels_1 = ax1.get_legend_handles_labels()
lines_2, labels_2 = ax2.get_legend_handles_labels()
ax1.legend(lines_1 + lines_2, labels_1 + labels_2, loc="best")

plt.title("Natural Validation Performance by Epoch")
plt.tight_layout()
plt.show()


## 11. Test Evaluation At Fixed And Validation-Selected Thresholds


In [ ]:
# Evaluate every saved checkpoint without tuning anything on the test set.
# Each checkpoint is reported at fixed 0.50 and at its validation-selected
# threshold. AUPRC and AUROC are threshold-free and therefore unchanged.


def safe_output_name(value):
    return (
        str(value)
        .lower()
        .replace(" ", "_")
        .replace("/", "_")
        .replace(".", "p")
    )


def evaluate_checkpoint_on_test(checkpoint_name, ckpt_path):
    if not ckpt_path.exists():
        print(f"Missing checkpoint for {checkpoint_name}: {ckpt_path}")
        return []

    ckpt = torch.load(
        ckpt_path,
        map_location=cfg.device,
        weights_only=False,
    )
    model.load_state_dict(ckpt["model_state"])
    model.to(cfg.device)
    model.eval()

    y_test, p_test, test_loss = predict_loader(test_loader)

    saved_threshold = float(ckpt.get("best_threshold", 0.5))
    threshold_specs = [("fixed_0.50", 0.5)]
    if not np.isclose(saved_threshold, 0.5):
        threshold_specs.append(
            ("validation_selected", saved_threshold)
        )

    print(f"\n=== {checkpoint_name} ===")
    print("checkpoint:", ckpt_path)
    print("run_id:", ckpt.get("run_id"))
    print("checkpoint epoch:", ckpt.get("best_epoch"))
    print("checkpoint selection_metric:", ckpt.get("selection_metric"))
    print("checkpoint best_val_auprc:", ckpt.get("best_val_auprc"))
    print("checkpoint best_val_auroc:", ckpt.get("best_val_auroc"))
    print("checkpoint best_val_loss:", ckpt.get("best_val_loss"))
    print("checkpoint validation-selected threshold:", saved_threshold)

    checkpoint_rows = []
    base_predictions = catalog[catalog.split == "test"].copy().reset_index(
        drop=True
    )
    base_predictions["pred_prob"] = p_test
    base_predictions["checkpoint_name"] = checkpoint_name

    for threshold_name, threshold in threshold_specs:
        test_metrics = classification_metrics(
            y_test,
            p_test,
            threshold=threshold,
        )
        test_metrics = {"loss_unweighted": test_loss, **test_metrics}

        print(f"\n{threshold_name}: threshold={threshold:.4f}")
        print("TEST:", test_metrics)

        cm = confusion_matrix(
            y_test,
            (p_test >= threshold).astype(int),
        )
        print("Confusion matrix:")
        print(cm)

        disp = ConfusionMatrixDisplay(
            confusion_matrix=cm,
            display_labels=["negative", "positive"],
        )
        fig, ax = plt.subplots(figsize=(5, 4))
        disp.plot(
            ax=ax,
            values_format="d",
            cmap="Blues",
            colorbar=False,
        )
        ax.set_title(
            f"{checkpoint_name}\n"
            f"{threshold_name}, threshold={threshold:.2f}"
        )
        plt.show()

        test_predictions = base_predictions.copy()
        test_predictions["threshold_name"] = threshold_name
        test_predictions["threshold"] = threshold
        test_predictions["pred_label"] = (
            test_predictions["pred_prob"] >= threshold
        ).astype(int)

        output_stem = (
            f"{safe_output_name(checkpoint_name)}_"
            f"{safe_output_name(threshold_name)}"
        )
        test_predictions.to_csv(
            model_dir / f"test_patch_predictions_{output_stem}.csv",
            index=False,
        )

        print("\nTest metrics by month:")
        month_rows = []
        for ym, group in test_predictions.groupby("ym"):
            yt = group["label"].to_numpy()
            yp = group["pred_prob"].to_numpy()
            month_row = {
                "ym": ym,
                "n": len(group),
                "positives": int((yt == 1).sum()),
                "negatives": int((yt == 0).sum()),
                **classification_metrics(
                    yt,
                    yp,
                    threshold=threshold,
                ),
            }
            month_rows.append(month_row)
            print(month_row)

        pd.DataFrame(month_rows).to_csv(
            model_dir / f"test_metrics_by_month_{output_stem}.csv",
            index=False,
        )

        checkpoint_rows.append({
            "checkpoint_name": checkpoint_name,
            "checkpoint_epoch": ckpt.get("best_epoch"),
            "checkpoint_selection_metric": ckpt.get("selection_metric"),
            "threshold_name": threshold_name,
            "threshold": threshold,
            **test_metrics,
        })

    return checkpoint_rows


checkpoint_specs = [
    ("best_selection", model_dir / "patch_cnn_best.pt"),
    (
        "best_val_recall_t05",
        model_dir / "patch_cnn_best_recall_t05.pt",
    ),
    (
        "best_val_f1_calibrated",
        model_dir / "patch_cnn_best_f1_calibrated.pt",
    ),
]

comparison_rows = []
for checkpoint_name, checkpoint_path in checkpoint_specs:
    comparison_rows.extend(
        evaluate_checkpoint_on_test(
            checkpoint_name,
            checkpoint_path,
        )
    )

if comparison_rows:
    comparison_df = pd.DataFrame(comparison_rows)
    display(comparison_df)
    comparison_df.to_csv(
        model_dir / "test_checkpoint_threshold_comparison.csv",
        index=False,
    )
    print(
        "saved comparison:",
        model_dir / "test_checkpoint_threshold_comparison.csv",
    )


In [ ]:
# Evaluate the best calibrated-F1 checkpoint for each test month
# using the threshold selected ONLY from validation.

ckpt_path = model_dir / "patch_cnn_best_f1_calibrated.pt"

ckpt = torch.load(
    ckpt_path,
    map_location=cfg.device,
    weights_only=False,
)

model.load_state_dict(ckpt["model_state"])
model.to(cfg.device)
model.eval()

validation_threshold = float(ckpt["best_threshold"])

print("Checkpoint epoch:", ckpt["best_epoch"])
print("Validation-selected threshold:", validation_threshold)

y_test, p_test, test_loss = predict_loader(test_loader)

test_predictions = (
    catalog[catalog["split"].eq("test")]
    .copy()
    .reset_index(drop=True)
)

if len(test_predictions) != len(y_test):
    raise RuntimeError(
        f"Catalog/prediction mismatch: {len(test_predictions)} != {len(y_test)}"
    )

test_predictions["pred_prob"] = p_test
test_predictions["pred_label"] = (
    test_predictions["pred_prob"] >= validation_threshold
).astype(int)

monthly_rows = []

for ym, group in test_predictions.groupby("ym", sort=True):
    y_true_month = group["label"].astype(int).to_numpy()
    y_prob_month = group["pred_prob"].to_numpy()
    y_pred_month = group["pred_label"].to_numpy()

    tn, fp, fn, tp = confusion_matrix(
        y_true_month,
        y_pred_month,
        labels=[0, 1],
    ).ravel()

    metrics = classification_metrics(
        y_true_month,
        y_prob_month,
        threshold=validation_threshold,
    )

    monthly_rows.append({
        "ym": ym,
        "threshold": validation_threshold,
        "samples": len(group),
        "positives": int((y_true_month == 1).sum()),
        "negatives": int((y_true_month == 0).sum()),
        "predicted_positive": int((y_pred_month == 1).sum()),
        "TN": int(tn),
        "FP": int(fp),
        "FN": int(fn),
        "TP": int(tp),
        "accuracy": metrics["accuracy"],
        "precision": metrics["precision"],
        "recall": metrics["recall"],
        "f1": metrics["f1"],
        "auroc": metrics["auroc"],
        "auprc": metrics["auprc"],
    })

    cm = np.array([[tn, fp], [fn, tp]])

    disp = ConfusionMatrixDisplay(
        confusion_matrix=cm,
        display_labels=["negative", "positive"],
    )

    fig, ax = plt.subplots(figsize=(4.5, 4))
    disp.plot(
        ax=ax,
        cmap="Blues",
        values_format="d",
        colorbar=False,
    )
    ax.set_title(
        f"{ym} Test Confusion Matrix\n"
        f"Validation threshold = {validation_threshold:.2f}"
    )
    plt.tight_layout()
    plt.show()

monthly_threshold_df = pd.DataFrame(monthly_rows)

display(monthly_threshold_df)

output_path = (
    model_dir
    / "test_metrics_by_month_validation_selected_threshold.csv"
)
monthly_threshold_df.to_csv(output_path, index=False)

print("Saved:", output_path)

### Expected thesis-level outputs

For the final thesis run, the selected calibrated-F1 checkpoint was evaluated on the September–December 2025 test period. The reported thesis values are:

- threshold selected from validation: `0.38`;
- test set: `2,202` patches;
- confusion matrix: `TN = 1,711`, `FP = 111`, `FN = 180`, `TP = 200`;
- accuracy `0.868`, precision `0.643`, recall `0.526`, F1 `0.579`;
- AUROC `0.814`, AUPRC `0.648`;
- random AUPRC baseline equal to test prevalence: `0.173`.

Checkpoint selection and threshold calibration use validation data only. Test-set tuning would invalidate the held-out evaluation.


## 12. Copy Outputs Back To Drive


In [ ]:
!rsync -ah --info=progress2 "$LOCAL_PROJECT_DIR/models/" "$DRIVE_PROJECT_DIR/models/"
!rsync -ah --info=progress2 "$LOCAL_PROJECT_DIR/prepared_data/" "$DRIVE_PROJECT_DIR/prepared_data/"


## 12. Optional wall-to-wall monthly classified-map export

The confusion matrices above summarize patch-level performance but do not show
where predictions occur. This section applies the validation-calibrated model
across every non-overlapping patch in each selected month and exports:

- a continuous positive-class probability GeoTIFF;
- a binary classified GeoTIFF using the validation-selected threshold;
- a three-panel PNG showing monthly XCH4 context, probability, and classified-positive patches over the study area, with known positive patch centers overlaid; and
- a CSV of predicted-positive patch-center coordinates.

These are **patch-resolution maps**, not pixel segmentations. Each output cell
represents one `cfg.patch_size x cfg.patch_size` model input patch.


In [ ]:
# Wall-to-wall monthly inference and classified-map export.
# Run this after the model, manifest, catalog, and normalization cells.

from affine import Affine
from matplotlib.colors import ListedColormap
from matplotlib.patches import Patch
from rasterio.transform import array_bounds, xy

MAP_SPLIT = "test"
MAP_MONTHS = None  # Example: ["2025-01", "2025-02"]; None exports every test month.
MAP_BATCH_SIZE = max(64, cfg.batch_size)
MIN_VALID_FRACTION = 0.80

classified_map_dir = model_dir / "classified_maps"
classified_map_dir.mkdir(parents=True, exist_ok=True)

map_ckpt_path = model_dir / "patch_cnn_best_f1_calibrated.pt"
map_ckpt = torch.load(
    map_ckpt_path,
    map_location=cfg.device,
    weights_only=False,
)
model.load_state_dict(map_ckpt["model_state"])
model.to(cfg.device)
model.eval()

map_threshold = float(map_ckpt["best_threshold"])
map_mean = np.asarray(
    map_ckpt.get("channel_mean", channel_mean),
    dtype=np.float32,
)[:, None, None]
map_std = np.asarray(
    map_ckpt.get("channel_std", channel_std),
    dtype=np.float32,
)[:, None, None]

print("Classified-map checkpoint:", map_ckpt_path)
print("Validation-selected threshold:", map_threshold)
print("Minimum valid-pixel fraction:", MIN_VALID_FRACTION)


class MonthlyScenePatchDataset(Dataset):
    def __init__(self, raster_row, patch_size, mean, std):
        self.raster_row = raster_row
        self.patch_size = int(patch_size)
        self.mean = mean
        self.std = std
        self.sources = {
            role: rasterio.open(getattr(raster_row, role))
            for role in CHANNEL_ROLES
        }
        ref = self.sources[CHANNEL_ROLES[0]]
        self.centers = list(
            non_overlapping_centers(
                ref.height,
                ref.width,
                self.patch_size,
            )
        )

    def __len__(self):
        return len(self.centers)

    def __getitem__(self, idx):
        row, col = self.centers[idx]
        half = self.patch_size // 2
        window = Window(
            col - half,
            row - half,
            self.patch_size,
            self.patch_size,
        )

        arrays = []
        for role in CHANNEL_ROLES:
            arr = self.sources[role].read(
                1,
                window=window,
                masked=True,
            ).filled(np.nan).astype(np.float32)
            arrays.append(arr)

        x_raw = np.stack(arrays, axis=0)
        valid_fraction = float(
            np.mean(np.all(np.isfinite(x_raw), axis=0))
        )
        x = (x_raw - self.mean) / self.std
        x = np.nan_to_num(
            x,
            nan=0.0,
            posinf=0.0,
            neginf=0.0,
        ).astype(np.float32)
        phys = np.zeros((N_PHYS,), dtype=np.float32)

        return (
            torch.from_numpy(x),
            torch.from_numpy(phys),
            int(row),
            int(col),
            valid_fraction,
        )

    def close(self):
        for src in self.sources.values():
            src.close()


def export_month_classified_map(raster_row):
    year = int(raster_row.year)
    month = int(raster_row.month)
    ym = f"{year}-{month:02d}"

    dataset = MonthlyScenePatchDataset(
        raster_row=raster_row,
        patch_size=cfg.patch_size,
        mean=map_mean,
        std=map_std,
    )
    loader = DataLoader(
        dataset,
        batch_size=MAP_BATCH_SIZE,
        shuffle=False,
        num_workers=0,
        pin_memory=True,
    )

    with rasterio.open(raster_row.xch4_mean) as ref:
        ref_profile = ref.profile.copy()
        ref_transform = ref.transform
        ref_crs = ref.crs
        grid_height = ref.height // cfg.patch_size
        grid_width = ref.width // cfg.patch_size

        # A display-resolution XCH4 background makes the classified patches
        # interpretable over the actual monthly study-area raster.
        display_scale = min(1.0, 1400.0 / max(ref.height, ref.width))
        display_height = max(1, int(round(ref.height * display_scale)))
        display_width = max(1, int(round(ref.width * display_scale)))
        xch4_background = ref.read(
            1,
            out_shape=(display_height, display_width),
            masked=True,
            resampling=Resampling.average,
        ).filled(np.nan).astype(np.float32)

    probability = np.full(
        (grid_height, grid_width),
        np.nan,
        dtype=np.float32,
    )

    try:
        with torch.no_grad():
            for x, phys, rows, cols, valid_fractions in tqdm(
                loader,
                desc=f"classify {ym}",
            ):
                x = x.to(cfg.device, non_blocking=True)
                phys = phys.to(cfg.device, non_blocking=True).float()
                probs = torch.sigmoid(model(x, phys)).cpu().numpy().ravel()

                rows = rows.numpy()
                cols = cols.numpy()
                valid_fractions = valid_fractions.numpy()

                for row, col, prob, valid_fraction in zip(
                    rows,
                    cols,
                    probs,
                    valid_fractions,
                ):
                    if valid_fraction < MIN_VALID_FRACTION:
                        continue
                    grid_row = (int(row) - cfg.patch_size // 2) // cfg.patch_size
                    grid_col = (int(col) - cfg.patch_size // 2) // cfg.patch_size
                    probability[grid_row, grid_col] = float(prob)
    finally:
        dataset.close()

    classified = np.full(
        probability.shape,
        255,
        dtype=np.uint8,
    )
    valid = np.isfinite(probability)
    classified[valid] = (
        probability[valid] >= map_threshold
    ).astype(np.uint8)

    map_transform = ref_transform * Affine.scale(
        cfg.patch_size,
        cfg.patch_size,
    )

    probability_path = (
        classified_map_dir / f"{ym}_positive_probability.tif"
    )
    classified_path = (
        classified_map_dir / f"{ym}_classified_threshold_{map_threshold:.2f}.tif"
    )
    png_path = classified_map_dir / f"{ym}_classified_map.png"
    points_path = (
        classified_map_dir / f"{ym}_predicted_positive_centers.csv"
    )

    probability_profile = ref_profile.copy()
    probability_profile.update(
        driver="GTiff",
        height=grid_height,
        width=grid_width,
        count=1,
        dtype="float32",
        nodata=-9999.0,
        transform=map_transform,
        crs=ref_crs,
        compress="deflate",
        predictor=2,
    )
    with rasterio.open(
        probability_path,
        "w",
        **probability_profile,
    ) as dst:
        dst.write(
            np.where(valid, probability, -9999.0).astype(np.float32),
            1,
        )

    classified_profile = ref_profile.copy()
    classified_profile.update(
        driver="GTiff",
        height=grid_height,
        width=grid_width,
        count=1,
        dtype="uint8",
        nodata=255,
        transform=map_transform,
        crs=ref_crs,
        compress="deflate",
    )
    with rasterio.open(
        classified_path,
        "w",
        **classified_profile,
    ) as dst:
        dst.write(classified, 1)

    positive_rows, positive_cols = np.where(classified == 1)
    center_x, center_y = xy(
        map_transform,
        positive_rows,
        positive_cols,
        offset="center",
    )
    predicted_positive_points = pd.DataFrame({
        "ym": ym,
        "grid_row": positive_rows,
        "grid_col": positive_cols,
        "x": center_x,
        "y": center_y,
        "crs": str(ref_crs),
        "pred_prob": probability[positive_rows, positive_cols],
        "threshold": map_threshold,
    }).sort_values("pred_prob", ascending=False)
    predicted_positive_points.to_csv(points_path, index=False)

    west, south, east, north = array_bounds(
        grid_height,
        grid_width,
        map_transform,
    )
    extent = (west, east, south, north)

    finite_xch4 = xch4_background[np.isfinite(xch4_background)]
    if finite_xch4.size:
        xch4_vmin, xch4_vmax = np.nanpercentile(
            finite_xch4,
            [2, 98],
        )
    else:
        xch4_vmin, xch4_vmax = 0.0, 1.0

    fig, axes = plt.subplots(
        1,
        3,
        figsize=(20, 6.5),
        constrained_layout=True,
        sharex=True,
        sharey=True,
    )

    background_kwargs = dict(
        extent=extent,
        origin="upper",
        cmap="Greys",
        vmin=xch4_vmin,
        vmax=xch4_vmax,
        interpolation="nearest",
    )

    xch4_image = axes[0].imshow(
        np.ma.masked_invalid(xch4_background),
        **background_kwargs,
    )
    axes[0].set_title(f"{ym} monthly XCH4 context")
    fig.colorbar(
        xch4_image,
        ax=axes[0],
        label="XCH4",
        fraction=0.046,
        pad=0.04,
    )

    axes[1].imshow(
        np.ma.masked_invalid(xch4_background),
        **background_kwargs,
    )
    prob_image = axes[1].imshow(
        np.ma.masked_invalid(probability),
        extent=extent,
        origin="upper",
        cmap="viridis",
        vmin=0.0,
        vmax=1.0,
        alpha=0.78,
        interpolation="nearest",
    )
    axes[1].set_title("Positive-class probability over study area")
    fig.colorbar(
        prob_image,
        ax=axes[1],
        label="Predicted probability",
        fraction=0.046,
        pad=0.04,
    )

    axes[2].imshow(
        np.ma.masked_invalid(xch4_background),
        **background_kwargs,
    )
    positive_only = np.ma.masked_where(classified != 1, classified)
    axes[2].imshow(
        positive_only,
        extent=extent,
        origin="upper",
        cmap=ListedColormap(["#d73027"]),
        vmin=0.5,
        vmax=1.5,
        alpha=0.82,
        interpolation="nearest",
    )
    axes[2].set_title(
        f"Classified positive patches (threshold={map_threshold:.2f})"
    )

    # Trace the monthly valid-data footprint on every panel.
    footprint = np.isfinite(xch4_background).astype(np.uint8)
    footprint_x = np.linspace(west, east, footprint.shape[1])
    footprint_y = np.linspace(north, south, footprint.shape[0])
    if footprint.min() != footprint.max():
        for ax in axes:
            ax.contour(
                footprint_x,
                footprint_y,
                footprint,
                levels=[0.5],
                colors="#222222",
                linewidths=0.7,
                alpha=0.9,
            )

    known_positive_patches = catalog[
        catalog["ym"].eq(ym) & catalog["label"].eq(1)
    ].drop_duplicates(["row", "col"])
    if not known_positive_patches.empty:
        known_x = []
        known_y = []
        for item in known_positive_patches.itertuples(index=False):
            x_coord, y_coord = ref_transform * (
                float(item.col),
                float(item.row),
            )
            known_x.append(x_coord)
            known_y.append(y_coord)
        for ax in axes[1:]:
            ax.scatter(
                known_x,
                known_y,
                s=38,
                facecolors="none",
                edgecolors="#00ffff",
                linewidths=1.2,
                label="Known positive patch",
            )

    legend_handles = [
        Patch(
            facecolor="#d73027",
            edgecolor="#d73027",
            alpha=0.82,
            label="Predicted positive patch",
        ),
    ]
    if not known_positive_patches.empty:
        legend_handles.append(
            plt.Line2D(
                [0],
                [0],
                marker="o",
                color="none",
                markerfacecolor="none",
                markeredgecolor="#00ffff",
                markersize=7,
                label="Known positive patch",
            )
        )
    axes[2].legend(
        handles=legend_handles,
        loc="lower left",
        framealpha=0.9,
    )

    axis_labels = (
        ("Longitude", "Latitude")
        if ref_crs is not None and ref_crs.is_geographic
        else ("X", "Y")
    )
    for ax in axes:
        ax.set_xlabel(axis_labels[0])
        ax.set_ylabel(axis_labels[1])
        ax.set_xlim(west, east)
        ax.set_ylim(south, north)

    fig.suptitle(
        f"{ym} methane-source classification over the study area\n"
        f"Patch model: {cfg.patch_size} x {cfg.patch_size} input pixels; "
        f"validation threshold: {map_threshold:.2f}"
    )
    fig.savefig(
        png_path,
        dpi=200,
        bbox_inches="tight",
    )
    plt.show()
    plt.close(fig)

    result = {
        "ym": ym,
        "valid_patches": int(valid.sum()),
        "predicted_positive_patches": int((classified == 1).sum()),
        "threshold": map_threshold,
        "probability_tif": str(probability_path),
        "classified_tif": str(classified_path),
        "map_png": str(png_path),
        "positive_centers_csv": str(points_path),
    }
    print(result)
    return result


map_manifest = manifest[manifest["split"].eq(MAP_SPLIT)].copy()
if MAP_MONTHS is not None:
    map_manifest = map_manifest[map_manifest["ym"].isin(MAP_MONTHS)]
map_manifest = map_manifest.sort_values(["year", "month"])

if map_manifest.empty:
    raise RuntimeError(
        f"No manifest rows found for split={MAP_SPLIT}, months={MAP_MONTHS}"
    )

classified_map_rows = []
for raster_row in map_manifest.itertuples(index=False):
    classified_map_rows.append(
        export_month_classified_map(raster_row)
    )

classified_map_summary = pd.DataFrame(classified_map_rows)
summary_path = classified_map_dir / "classified_map_summary.csv"
classified_map_summary.to_csv(summary_path, index=False)
display(classified_map_summary)
print("Saved classified-map summary:", summary_path)


## 13. Held-out confusion-matrix patch maps

This section loads the saved validation-selected checkpoint and maps the exact
test patches represented by the confusion matrix. It requires the data and
model definitions initialized through Section 9; model training and the general
evaluation section need not be repeated when a compatible checkpoint is
available.

Each test patch is mapped as TN, FP, FN, or TP. The final assertion verifies
that monthly totals equal the overall confusion matrix. The threshold is fixed
from validation data, and all mapped samples belong to the held-out test split.


In [ ]:
# Monthly spatial maps of the exact samples behind the confusion matrix.
# Requires the model, data loaders, prediction loader, and evaluation helpers.

from matplotlib.colors import ListedColormap, BoundaryNorm
from matplotlib.patches import Patch
from rasterio.transform import array_bounds
from affine import Affine

CONFUSION_MAP_SPLIT = "test"

required_runtime_names = [
    "cfg",
    "model",
    "catalog",
    "manifest",
    "test_loader",
    "val_loader",
]
missing_runtime_names = [
    name
    for name in required_runtime_names
    if name not in globals()
]
if missing_runtime_names:
    raise RuntimeError(
        "Required runtime objects are missing: "
        + ", ".join(missing_runtime_names)
        + ". Run the notebook from Section 1 through "
        + "Section 9 (CNN Classifier) "
        + "before executing this section."
    )


def find_saved_confusion_checkpoint():
    checkpoint_name = "patch_cnn_best_f1_calibrated.pt"
    candidates = []

    if "model_dir" in globals():
        candidates.append(Path(model_dir) / checkpoint_name)

    experiment_name = (
        "multiscale_methane_strict_negative_2023_2025_"
        f"ps{cfg.patch_size}"
    )
    project_roots = []
    for value in [
        globals().get("PROJECT_DIR"),
        globals().get("LOCAL_PROJECT_DIR"),
        globals().get("DRIVE_PROJECT_DIR"),
        cfg.project_dir,
        "/content/CNN_point_source",
        "/content/drive/MyDrive/CNN_point_source",
    ]:
        if value:
            root = Path(value)
            if root not in project_roots:
                project_roots.append(root)

    for root in project_roots:
        run_root = root / "models" / experiment_name
        configured = run_root / str(cfg.run_id) / checkpoint_name
        candidates.append(configured)
        if run_root.exists():
            candidates.extend(run_root.glob(f"*/{checkpoint_name}"))

    existing = []
    for candidate in candidates:
        try:
            if candidate.exists() and candidate not in existing:
                existing.append(candidate)
        except OSError:
            continue

    if not existing:
        searched = "\n".join(str(path) for path in candidates)
        raise FileNotFoundError(
            "Saved calibrated checkpoint was not found. "
            "Confirm Drive is mounted and the models folder was "
            "copied with rsync. Searched:\n" + searched
        )

    configured_matches = [
        path
        for path in existing
        if path.parent.name == str(cfg.run_id)
    ]
    if configured_matches:
        return configured_matches[0]

    return max(
        existing,
        key=lambda path: path.stat().st_mtime,
    )


CONFUSION_MAP_CHECKPOINT = find_saved_confusion_checkpoint()
model_dir = CONFUSION_MAP_CHECKPOINT.parent
confusion_map_dir = model_dir / "monthly_confusion_maps"
confusion_map_dir.mkdir(parents=True, exist_ok=True)

print("Using saved run directory:", model_dir)
print("Using checkpoint:", CONFUSION_MAP_CHECKPOINT)

cm_ckpt = torch.load(
    CONFUSION_MAP_CHECKPOINT,
    map_location=cfg.device,
    weights_only=False,
)
model.load_state_dict(cm_ckpt["model_state"])
model.to(cfg.device)
model.eval()

cm_threshold = float(cm_ckpt["best_threshold"])

if CONFUSION_MAP_SPLIT == "test":
    cm_loader = test_loader
elif CONFUSION_MAP_SPLIT == "val":
    cm_loader = val_loader
else:
    raise ValueError(
        "CONFUSION_MAP_SPLIT must be 'test' or 'val'."
    )


@torch.no_grad()
def predict_confusion_map_loader(loader):
    model.eval()
    y_true = []
    y_prob = []

    for x, phys, y in tqdm(
        loader,
        desc=f"predict {CONFUSION_MAP_SPLIT}",
    ):
        x = x.to(cfg.device, non_blocking=True)
        phys = phys.to(cfg.device, non_blocking=True).float()
        logits = model(x, phys)
        prob = torch.sigmoid(logits)
        y_true.append(y.numpy().ravel())
        y_prob.append(prob.cpu().numpy().ravel())

    return (
        np.concatenate(y_true) if y_true else np.array([]),
        np.concatenate(y_prob) if y_prob else np.array([]),
    )


cm_y_true, cm_y_prob = predict_confusion_map_loader(cm_loader)
cm_predictions = (
    catalog[catalog["split"].eq(CONFUSION_MAP_SPLIT)]
    .copy()
    .reset_index(drop=True)
)

if len(cm_predictions) != len(cm_y_true):
    raise RuntimeError(
        "Catalog/prediction mismatch: "
        f"{len(cm_predictions)} != {len(cm_y_true)}"
    )

cm_predictions["pred_prob"] = cm_y_prob
cm_predictions["pred_label"] = (
    cm_predictions["pred_prob"] >= cm_threshold
).astype(np.uint8)

cm_predictions["outcome"] = np.select(
    [
        (cm_predictions["label"] == 0)
        & (cm_predictions["pred_label"] == 0),
        (cm_predictions["label"] == 0)
        & (cm_predictions["pred_label"] == 1),
        (cm_predictions["label"] == 1)
        & (cm_predictions["pred_label"] == 0),
        (cm_predictions["label"] == 1)
        & (cm_predictions["pred_label"] == 1),
    ],
    ["TN", "FP", "FN", "TP"],
    default="UNKNOWN",
)

OUTCOME_ORDER = ["TN", "FP", "FN", "TP"]
OUTCOME_CODE = {"TN": 0, "FP": 1, "FN": 2, "TP": 3}
OUTCOME_COLORS = {
    "TN": "#4575b4",
    "FP": "#fdae61",
    "FN": "#984ea3",
    "TP": "#1a9850",
}
outcome_cmap = ListedColormap(
    [OUTCOME_COLORS[name] for name in OUTCOME_ORDER]
)
outcome_norm = BoundaryNorm(
    [-0.5, 0.5, 1.5, 2.5, 3.5],
    outcome_cmap.N,
)


def confusion_counts(frame):
    counts = frame["outcome"].value_counts()
    return {
        name: int(counts.get(name, 0))
        for name in OUTCOME_ORDER
    }


def monthly_confusion_grid(month_group, raster_row):
    with rasterio.open(raster_row.xch4_mean) as ref:
        ref_transform = ref.transform
        ref_crs = ref.crs
        ref_height = ref.height
        ref_width = ref.width

        display_scale = min(
            1.0,
            1400.0 / max(ref_height, ref_width),
        )
        display_height = max(
            1,
            int(round(ref_height * display_scale)),
        )
        display_width = max(
            1,
            int(round(ref_width * display_scale)),
        )
        xch4_background = ref.read(
            1,
            out_shape=(display_height, display_width),
            masked=True,
            resampling=Resampling.average,
        ).filled(np.nan).astype(np.float32)

    grid_height = ref_height // cfg.patch_size
    grid_width = ref_width // cfg.patch_size
    outcome_grid = np.full(
        (grid_height, grid_width),
        np.nan,
        dtype=np.float32,
    )

    for item in month_group.itertuples(index=False):
        grid_row = (
            int(item.row) - cfg.patch_size // 2
        ) // cfg.patch_size
        grid_col = (
            int(item.col) - cfg.patch_size // 2
        ) // cfg.patch_size
        if not (
            0 <= grid_row < grid_height
            and 0 <= grid_col < grid_width
        ):
            raise RuntimeError(
                f"Patch center outside map grid: {item.ym}, "
                f"row={item.row}, col={item.col}"
            )
        if np.isfinite(outcome_grid[grid_row, grid_col]):
            raise RuntimeError(
                f"Duplicate evaluated grid cell: {item.ym}, "
                f"grid_row={grid_row}, grid_col={grid_col}"
            )
        outcome_grid[grid_row, grid_col] = OUTCOME_CODE[item.outcome]

    map_transform = ref_transform * Affine.scale(
        cfg.patch_size,
        cfg.patch_size,
    )
    west, south, east, north = array_bounds(
        grid_height,
        grid_width,
        map_transform,
    )

    return {
        "outcome_grid": outcome_grid,
        "xch4_background": xch4_background,
        "extent": (west, east, south, north),
        "crs": ref_crs,
    }


def draw_monthly_confusion_map(
    ax,
    month_group,
    raster_row,
    title=None,
    show_legend=True,
):
    ym = str(month_group["ym"].iloc[0])
    map_data = monthly_confusion_grid(
        month_group,
        raster_row,
    )
    west, east, south, north = map_data["extent"]

    xch4_background = map_data["xch4_background"]
    finite_background = xch4_background[
        np.isfinite(xch4_background)
    ]
    if finite_background.size:
        bg_vmin, bg_vmax = np.nanpercentile(
            finite_background,
            [2, 98],
        )
    else:
        bg_vmin, bg_vmax = 0.0, 1.0

    ax.imshow(
        np.ma.masked_invalid(xch4_background),
        extent=map_data["extent"],
        origin="upper",
        cmap="Greys",
        vmin=bg_vmin,
        vmax=bg_vmax,
        interpolation="nearest",
    )
    ax.imshow(
        np.ma.masked_invalid(map_data["outcome_grid"]),
        extent=map_data["extent"],
        origin="upper",
        cmap=outcome_cmap,
        norm=outcome_norm,
        alpha=0.88,
        interpolation="nearest",
    )

    counts = confusion_counts(month_group)
    ax.set_title(
        title
        or (
            f"{ym}  "
            f"TN={counts['TN']}  FP={counts['FP']}  "
            f"FN={counts['FN']}  TP={counts['TP']}"
        ),
        fontsize=10,
    )
    ax.set_xlim(west, east)
    ax.set_ylim(south, north)
    ax.set_xlabel(
        "Longitude"
        if map_data["crs"] is not None
        and map_data["crs"].is_geographic
        else "X"
    )
    ax.set_ylabel(
        "Latitude"
        if map_data["crs"] is not None
        and map_data["crs"].is_geographic
        else "Y"
    )

    if show_legend:
        handles = [
            Patch(
                facecolor=OUTCOME_COLORS[name],
                edgecolor="none",
                label=f"{name}: {counts[name]}",
            )
            for name in OUTCOME_ORDER
        ]
        ax.legend(
            handles=handles,
            loc="lower left",
            framealpha=0.92,
            fontsize=8,
            ncol=2,
        )

    return counts


manifest_by_ym = (
    manifest.set_index("ym", drop=False)
    .to_dict("index")
)
monthly_count_rows = []

for ym, month_group in cm_predictions.groupby(
    "ym",
    sort=True,
):
    raster_row = type(
        "ManifestRow",
        (),
        manifest_by_ym[ym],
    )()
    counts = confusion_counts(month_group)

    fig, ax = plt.subplots(
        figsize=(11.5, 6.5),
        constrained_layout=True,
    )
    draw_monthly_confusion_map(
        ax,
        month_group,
        raster_row,
        show_legend=True,
    )
    fig.suptitle(
        "Exact catalog patches used in the confusion matrix\n"
        f"split={CONFUSION_MAP_SPLIT}, "
        f"validation-selected threshold={cm_threshold:.2f}"
    )
    month_path = (
        confusion_map_dir
        / f"{ym}_{CONFUSION_MAP_SPLIT}_TN_FP_FN_TP.png"
    )
    fig.savefig(
        month_path,
        dpi=220,
        bbox_inches="tight",
    )
    plt.show()
    plt.close(fig)

    monthly_count_rows.append({
        "ym": ym,
        "samples": len(month_group),
        "threshold": cm_threshold,
        **counts,
        "map_png": str(month_path),
    })

monthly_confusion_counts = pd.DataFrame(
    monthly_count_rows
).sort_values("ym")

# Compact all-month overview.
months = monthly_confusion_counts["ym"].tolist()
ncols = 3
nrows = int(np.ceil(len(months) / ncols))
overview_fig, overview_axes = plt.subplots(
    nrows,
    ncols,
    figsize=(18, 5.2 * nrows),
    constrained_layout=True,
    squeeze=False,
)

for ax, ym in zip(overview_axes.ravel(), months):
    month_group = cm_predictions[
        cm_predictions["ym"].eq(ym)
    ]
    raster_row = type(
        "ManifestRow",
        (),
        manifest_by_ym[ym],
    )()
    draw_monthly_confusion_map(
        ax,
        month_group,
        raster_row,
        show_legend=False,
    )

for ax in overview_axes.ravel()[len(months):]:
    ax.axis("off")

overview_handles = [
    Patch(
        facecolor=OUTCOME_COLORS[name],
        edgecolor="none",
        label=name,
    )
    for name in OUTCOME_ORDER
]
overview_fig.legend(
    handles=overview_handles,
    loc="lower center",
    ncol=4,
    framealpha=0.95,
)
overview_fig.suptitle(
    "Monthly spatial outcomes for the exact confusion-matrix patches\n"
    f"split={CONFUSION_MAP_SPLIT}, "
    f"validation-selected threshold={cm_threshold:.2f}",
    fontsize=16,
)
overview_path = (
    confusion_map_dir
    / f"all_months_{CONFUSION_MAP_SPLIT}_TN_FP_FN_TP.png"
)
overview_fig.savefig(
    overview_path,
    dpi=220,
    bbox_inches="tight",
)
plt.show()
plt.close(overview_fig)

overall_matrix = confusion_matrix(
    cm_predictions["label"].astype(int),
    cm_predictions["pred_label"].astype(int),
    labels=[0, 1],
)
overall_tn, overall_fp, overall_fn, overall_tp = (
    overall_matrix.ravel()
)
overall_from_matrix = {
    "TN": int(overall_tn),
    "FP": int(overall_fp),
    "FN": int(overall_fn),
    "TP": int(overall_tp),
}
overall_from_maps = {
    name: int(monthly_confusion_counts[name].sum())
    for name in OUTCOME_ORDER
}

assert overall_from_maps == overall_from_matrix, (
    "Monthly map totals do not match the confusion matrix: "
    f"maps={overall_from_maps}, matrix={overall_from_matrix}"
)
assert sum(overall_from_maps.values()) == len(cm_predictions)

counts_path = (
    confusion_map_dir
    / f"monthly_{CONFUSION_MAP_SPLIT}_confusion_counts.csv"
)
predictions_path = (
    confusion_map_dir
    / f"exact_{CONFUSION_MAP_SPLIT}_patch_outcomes.csv"
)
monthly_confusion_counts.to_csv(counts_path, index=False)
cm_predictions.to_csv(predictions_path, index=False)

display(monthly_confusion_counts)
print("Overall confusion matrix:")
print(overall_matrix)
print("Overall TN/FP/FN/TP:", overall_from_matrix)
print("Verified: monthly map totals exactly match the confusion matrix.")
print("All-month overview:", overview_path)
print("Monthly counts:", counts_path)
print("Exact mapped patch outcomes:", predictions_path)
